# NCBI Assembly Promote & Archive (Phase 3)

Promotes staged assembly files from S3 staging (written by CTS Phase 2)
to their final Lakehouse paths, archives replaced/suppressed assemblies,
and trims the transfer manifest for resumability.

Steps:
1. Configure staging prefix, removed manifest, updated manifest, and release tag
2. Scan staged files and display summary
3. Archive existing versions of updated assemblies (pre-overwrite)
4. Promote files to final paths with MD5 metadata
5. Archive replaced/suppressed assemblies
6. Trim manifest (remove promoted entries)

## Path formats quick reference

| Suffix in variable name | Format | Example |
|-------------------------|--------|---------|
| `_BUCKET` | bucket name only | `cdm-lake` |
| `_KEY_PREFIX` | S3 key prefix (no scheme/bucket) | `staging/run1/` |
| `_S3_KEY` | S3 object key (no scheme/bucket) | `staging/transfer_manifest.txt` |
| `_PATH` | local filesystem path | `output/removed_manifest.txt` |

Lakehouse object: `s3://{STORE_BUCKET}/{LAKEHOUSE_KEY_PREFIX}raw_data/…/{filename}`
Staging object:   `s3://{STORE_BUCKET}/{STAGING_KEY_PREFIX}raw_data/…/{filename}`

In [ ]:
"""Imports and S3 client initialisation."""

import json

from cdm_data_loaders.ncbi_ftp.promote import (
    DEFAULT_LAKEHOUSE_KEY_PREFIX,
    promote_from_s3,
)
from cdm_data_loaders.utils.s3 import get_s3_client

In [ ]:
"""Configure parameters.

Path layout (how variables compose into a full S3 object path):
  s3://{STORE_BUCKET}/{LAKEHOUSE_KEY_PREFIX}raw_data/{GCF|GCA}/{nnn}/{nnn}/{nnn}/{assembly_dir}/{file}
  s3://{STORE_BUCKET}/{STAGING_KEY_PREFIX}raw_data/{assembly_dir}/{file}
"""

# S3 bucket where staged files and final Lakehouse data live
# format: bucket name (no s3:// scheme)
STORE_BUCKET = "cdm-lake"

# Staging prefix written by CTS Phase 2
# format: S3 key prefix within STORE_BUCKET (no scheme, no bucket)
STAGING_KEY_PREFIX = "staging/run1/"

# Local path to removed_manifest.txt from Phase 1 (or None to skip archiving)
# format: local file path
REMOVED_MANIFEST_PATH: str | None = None  # e.g. "output/removed_manifest.txt"

# Local path to updated_manifest.txt from Phase 1 (or None to skip pre-overwrite archiving)
# format: local file path
UPDATED_MANIFEST_PATH: str | None = None  # e.g. "output/updated_manifest.txt"

# NCBI release tag for archive metadata (e.g. "2024-01")
NCBI_RELEASE: str | None = None

# S3 key of transfer_manifest.txt for trimming after promotion (or None to skip).
# Only needed if the manifest was uploaded to S3 (e.g. via the staging cell in Phase 1).
# format: S3 object key within STORE_BUCKET (no scheme, no bucket)
MANIFEST_S3_KEY: str | None = "staging/run1/input/transfer_manifest.txt"  # e.g. "staging/transfer_manifest.txt"

# Final Lakehouse path prefix
# format: S3 key prefix within STORE_BUCKET (no scheme, no bucket)
LAKEHOUSE_KEY_PREFIX = DEFAULT_LAKEHOUSE_KEY_PREFIX

# Dry-run mode — log actions without making changes
DRY_RUN = False

print(f"Bucket:              {STORE_BUCKET}")
print(f"Staging key prefix:  {STAGING_KEY_PREFIX}")
print(f"Updated manifest:    {UPDATED_MANIFEST_PATH}")
print(f"NCBI release:        {NCBI_RELEASE}")
print(f"Manifest S3 key:     {MANIFEST_S3_KEY}")
print(f"Lakehouse prefix:    {LAKEHOUSE_KEY_PREFIX}")
print(f"Dry-run:             {DRY_RUN}")

In [ ]:
"""Scan staged files and display summary."""

s3 = get_s3_client()
paginator = s3.get_paginator("list_objects_v2")

staged: list[str] = []
for page in paginator.paginate(Bucket=STORE_BUCKET, Prefix=STAGING_KEY_PREFIX):
    staged.extend(obj["Key"] for obj in page.get("Contents", []))

sidecars = [k for k in staged if k.endswith((".md5", ".crc64nvme"))]
data_files = [k for k in staged if k not in set(sidecars)]

print(f"Staged objects: {len(staged)}")
print(f"  Data files:   {len(data_files)}")
print(f"  Sidecars:     {len(sidecars)}")

# Show first few data files
PREVIEW_COUNT = 10
for key in data_files[:PREVIEW_COUNT]:
    print(f"  {key}")
if len(data_files) > PREVIEW_COUNT:
    print(f"  ... and {len(data_files) - PREVIEW_COUNT} more")

In [ ]:
"""Promote staged files to final Lakehouse paths."""

report = promote_from_s3(
    staging_key_prefix=STAGING_KEY_PREFIX,
    bucket=STORE_BUCKET,
    removed_manifest_path=REMOVED_MANIFEST_PATH,
    updated_manifest_path=UPDATED_MANIFEST_PATH,
    ncbi_release=NCBI_RELEASE,
    manifest_s3_key=MANIFEST_S3_KEY,
    lakehouse_key_prefix=LAKEHOUSE_KEY_PREFIX,
    dry_run=DRY_RUN,
)

print(json.dumps(report, indent=2))

In [ ]:
"""Display promotion report."""

print("=" * 50)
print("PROMOTION REPORT")
print("=" * 50)
print(f"Promoted:  {report['promoted']}")
print(f"Archived:  {report['archived']}")
print(f"Failed:    {report['failed']}")
print(f"Dry-run:   {report['dry_run']}")
print(f"Timestamp: {report['timestamp']}")

if report["failed"] > 0:
    print("\n⚠️  Some operations failed — check logs above for details.")

if report["dry_run"]:
    print("\n📋 This was a dry-run. Set DRY_RUN = False and re-run to apply changes.")

In [ ]:
"""Inspect frictionless descriptors written to metadata/."""

from cdm_data_loaders.ncbi_ftp.metadata import build_descriptor_key

s3 = get_s3_client()
paginator = s3.get_paginator("list_objects_v2")

descriptor_keys: list[str] = []
for page in paginator.paginate(Bucket=STORE_BUCKET, Prefix=LAKEHOUSE_KEY_PREFIX + "metadata/"):
    descriptor_keys.extend(obj["Key"] for obj in page.get("Contents", []))

print(f"Found {len(descriptor_keys)} descriptor(s) in metadata/")

for key in descriptor_keys[:5]:  # preview first 5
    obj = s3.get_object(Bucket=STORE_BUCKET, Key=key)
    descriptor = json.loads(obj["Body"].read())
    print()
    print(f"  Key:        {key}")
    print(f"  Identifier: {descriptor.get('identifier')}")
    print(f"  Version:    {descriptor.get('version')}")
    print(f"  Resources:  {len(descriptor.get('resources', []))} file(s)")

if len(descriptor_keys) > 5:
    print(f"  ... and {len(descriptor_keys) - 5} more")